In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path

import importlib
import track

# Development helper: uncomment after editing track.py.
# importlib.reload(track)

# Load Snapshot

按区域窗口加载单日 OFES NP30 快照。MOM3 Arakawa B-grid 的 u/v 均由四角共置到示踪物中心；参照 JAMSTEC 对公开 OFES DODS 的通用错误 mbar 警告，结合交付层位将 lev 按深度米解释，w 保留在层界面。

In [ ]:
snap = track.load_ofes_snapshot(
    '2003-04-05', lon_bounds=(142, 150), lat_bounds=(32, 39),
    depth_bounds=(0, 1100),
)
print(list(snap.keys()))
print(f"do2: {snap['do2'].shape}, u: {snap['u'].shape}, w: {snap['w'].shape}")

# Quick-Look Horizontal Slice

In [ ]:
track.plot_ofes_snapshot_quick(snap, variable='do2', depth=600.0)

# Vertical Profile Extraction

从快照中双线性插值提取定深虚拟剖面；Depth / do2 / temp / salinity 继续送入通用单剖面 detector。

In [ ]:
prof = track.extract_ofes_profile_interp(snap, lon=144.5, lat=35.0,
                                         variables=['do2', 'temp', 'salinity'])
prof.head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 6), sharey=True)
for ax, var, label in zip(axes, ['do2', 'temp', 'salinity'],
                          ['DO₂ (μmol kg⁻¹)', 'Potential temp (°C)', 'Salinity (PSS-78)']):
    ax.plot(prof[var], prof['Depth'], linewidth=1.2)
    ax.set_xlabel(label)
    ax.invert_yaxis()
axes[0].set_ylabel('Depth (m)')
fig.suptitle(f"OFES profile  144.5°E, 35°N  {snap['date'].strftime('%Y-%m-%d')}")
fig.tight_layout()

# δDO Detection on OFES

复用观测 pipeline 的 `calculate_delta_do` 在 OFES 虚拟剖面上检测异常。

In [ ]:
det_cfg = track.make_detection_config('do')

result = track.detect_ofes_delta_do(snap, lon=144.5, lat=35.0,
                                    detection_config=det_cfg)
result

In [ ]:
det_prof = track.extract_ofes_profile_interp(snap, lon=144.5, lat=35.0,
                                             variables=['do2'])
fig, ax = plt.subplots(figsize=(5, 7))
ax.plot(det_prof['do2'], det_prof['Depth'], 'b-', linewidth=1.2)
ax.axhline(det_cfg.anomaly_min_depth, color='gray', ls='--', alpha=0.5,
           label=f'min depth {det_cfg.anomaly_min_depth:.0f} m')
for _, row in result.iterrows():
    ax.plot(row['do_value'], row['depth'], 'ro', ms=7)
    ax.annotate(f"ΔDO={row['delta_do']:.1f}", (row['do_value'], row['depth']),
                textcoords='offset points', xytext=(8, 0), color='red', fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('DO₂ (μmol kg⁻¹)')
ax.set_ylabel('Depth (m)')
ax.set_title(f"δDO detection  144.5°E, 35°N  {snap['date'].strftime('%Y-%m-%d')}")
ax.legend(loc='lower left')
fig.tight_layout()

# Annual Event Catalog

年度 ΔDO20/35/50 扫描已经通过 parity 与掩码审计。这里仅读取固定运行，不在 notebook 中重扫或重排候选。

In [ ]:
catalog_run = Path(
    'plot_outputs/do/ofes_np30_ke/ofes_delta_do_catalog/'
    '20030101_20031231_cf957935d38a'
)
catalog_manifest = json.loads((catalog_run / 'manifest.json').read_text())
if (
    catalog_manifest.get('schema_version') != 2
    or catalog_manifest.get('status') != 'complete'
):
    raise RuntimeError('Use a completed depth/m schema-v2 OFES catalog.')
event_catalog = pd.read_parquet(catalog_run / 'event_catalog.parquet')
{
    'status': catalog_manifest['status'],
    'days': catalog_manifest['completed_days'],
    'events': len(event_catalog),
}

# Ranked Water-Mass Diagnostics

正式候选固定为质量排名前五的 DO50 events。等密度水团、heave、动力量与负对照均从已验证诊断表读取，500–900 m 排名只作敏感性分析。

In [ ]:
diagnostic_run = catalog_run / (
    'event_diagnostics/ofes_events_21efbe902ab7'
)
diagnostic_manifest = json.loads((diagnostic_run / 'manifest.json').read_text())
if (
    diagnostic_manifest.get('schema_version') != 2
    or diagnostic_manifest.get('status') != 'complete'
):
    raise RuntimeError('Use completed diagnostics from the schema-v2 catalog.')
selected_events = pd.read_parquet(
    diagnostic_run / 'selected_events.parquet'
)
event_diagnostics = pd.read_parquet(
    diagnostic_run / 'event_diagnostic_summary.parquet'
)
event_diagnostics

# Event Evolution Diagnostics

读取已验证的 schema-v2 演化运行：每个固定候选包含 ±10 天语境、start/peak/end 等密度面图，并共享年度总览、负对照和 Hosoda 三日期集成检验。如需重建，调用 `track.build_ofes_event_evolution_diagnostics`。

In [ ]:
evolution_run = diagnostic_run / (
    'event_evolution/ofes_evolution_46b47184fd21'
)
evolution_manifest = json.loads((evolution_run / 'manifest.json').read_text())
if (
    evolution_manifest.get('schema_version') != 2
    or evolution_manifest.get('status') != 'complete'
):
    raise RuntimeError('Use a completed schema-v2 OFES evolution run.')
evolution_manifest['outputs']['figures']

# Trajectory Gate

轨迹是否启动由正式 gate 文件决定。当前不会把 depth-varying 的 Eulerian ΔDO 对象强行解释成物质粒子，也不会用尚未验证的日场时间插值或三维 w 耦合支持来源归因。

In [ ]:
trajectory_gate = evolution['trajectory_gate']
trajectory_gate